# PulseSchedule to QuEL-3 Sequencer Flow

This notebook demonstrates how a `PulseSchedule` is converted into a QuEL-3 execution payload and then compiled into a `quelware-client`-compatible sequencer.

The flow is:

1. Build a `PulseSchedule` and `MeasurementSchedule`
2. Build `Quel3ExecutionPayload` with `Quel3MeasurementBackendAdapter`
3. Compile payload into sequencer events/waveforms with `Quel3SequencerCompiler`
4. Visualize both `MeasurementSchedule` and sequencer timelines with shared plotting helpers


In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from types import SimpleNamespace
from typing import Any, cast

import numpy as np
from qxpulse import Blank, Gaussian, PulseSchedule, set_sampling_period

from qubex.analysis.visalization.schedule_visualizer import (
    plot_measurement_schedule,
    plot_sequencer_timeline,
)
from qubex.backend.quel3 import Quel3ExecutionPayload, Quel3SequencerCompiler
from qubex.measurement.adapters import Quel3MeasurementBackendAdapter
from qubex.measurement.measurement_constraint_profile import (
    MeasurementConstraintProfile,
)
from qubex.measurement.models.capture_schedule import Capture, CaptureSchedule
from qubex.measurement.models.measurement_config import (
    DspConfig,
    FrequencyConfig,
    MeasurementConfig,
)
from qubex.measurement.models.measurement_schedule import MeasurementSchedule

In [ ]:
# QuEL-3 default schedule dt
# NOTE: PulseSchedule currently assumes one common dt across channels.
DT_NS = 0.4
READOUT_EXPORT_SAMPLING_PERIOD_FS = 800_000

# Apply notebook-wide default sampling period.
set_sampling_period(DT_NS)

# Build shape-equivalent control pulses (different only by scale/phase).
base = Gaussian(
    duration=8.0,
    amplitude=0.5,
    sigma=1.6,
    zero_bounds=False,
)
variant = base.scaled(0.6).shifted(np.deg2rad(35.0))

# Build one readout waveform prototype with the same schedule dt.
readout = Gaussian(
    duration=8.0,
    amplitude=0.35,
    sigma=1.6,
    zero_bounds=False,
)

with PulseSchedule(["Q00", "Q01", "RQ00"]) as pulse_schedule:
    # Control block
    pulse_schedule.add("Q00", Blank(duration=4.0))
    pulse_schedule.add("Q00", base)

    pulse_schedule.add("Q01", Blank(duration=4.0))
    pulse_schedule.add("Q01", variant)

    # Align channels before readout
    pulse_schedule.barrier()

    # Readout block (multi-round)
    pulse_schedule.add("RQ00", readout)
    pulse_schedule.add("RQ00", Blank(duration=8.0))
    pulse_schedule.add("RQ00", readout.scaled(0.9))

print("pulse_schedule.duration_ns =", pulse_schedule.duration)
print("pulse_schedule.labels =", pulse_schedule.labels)

pulse_schedule.plot()

In [ ]:
measurement_schedule = MeasurementSchedule(
    pulse_schedule=pulse_schedule,
    capture_schedule=CaptureSchedule(
        captures=[
            Capture(
                channels=["RQ00"],
                start_time=12.0,
                duration=8.0,
            ),
            Capture(
                channels=["RQ00"],
                start_time=28.0,
                duration=8.0,
            ),
        ]
    ),
)

plot_measurement_schedule(
    measurement_schedule,
    title="MeasurementSchedule Timeline (Before Adapter)",
)

In [ ]:
class _BackendControllerStub:
    DEFAULT_SAMPLING_PERIOD = DT_NS

    def __init__(self) -> None:
        self._alias_map = {
            "Q00": "ctrl-00",
            "Q01": "ctrl-01",
            "RQ00": "readout-00",
        }

    def resolve_instrument_alias(self, target: str) -> str:
        return self._alias_map.get(target, target)


class _ExperimentSystemStub:
    @staticmethod
    def get_awg_frequency(target: str) -> float:
        return {
            "Q00": 5.00e9,
            "Q01": 5.10e9,
            "RQ00": 6.80e9,
        }[target]


measurement_config = MeasurementConfig(
    mode="avg",
    shots=16,
    interval=100.0,
    frequency=FrequencyConfig(frequencies={}),
    dsp=DspConfig(
        enable_dsp_demodulation=True,
        enable_dsp_sum=False,
        enable_dsp_classification=False,
        line_param0=(1.0, 0.0, 0.0),
        line_param1=(0.0, 1.0, 0.0),
    ),
)

adapter = Quel3MeasurementBackendAdapter(
    backend_controller=_BackendControllerStub(),
    experiment_system=cast(Any, _ExperimentSystemStub()),
    constraint_profile=MeasurementConstraintProfile.relaxed(sampling_period_ns=DT_NS),
)
request = adapter.build_execution_request(
    schedule=measurement_schedule,
    config=measurement_config,
)

payload = request.payload
assert isinstance(payload, Quel3ExecutionPayload)
print(type(payload).__name__)
print(
    "interval_ns=",
    payload.interval_ns,
    "repeats=",
    payload.repeats,
    "mode=",
    payload.mode,
)

In [ ]:
for target, timeline in payload.timelines.items():
    print(f"\n[{target}] alias={payload.instrument_aliases[target]}")
    print(
        f"  timeline_dt={timeline.sampling_period_ns} ns "
        f"length={timeline.length_ns} ns modulation_hz={timeline.modulation_frequency_hz}"
    )
    for idx, event in enumerate(timeline.events):
        print(
            f"  event[{idx}] start={event.start_offset_ns:>4.1f} ns "
            f"dt={event.sampling_period_ns} ns "
            f"samples={len(event.waveform):>2d} "
            f"peak={np.max(np.abs(event.waveform)):.4f}"
        )
    for window in timeline.capture_windows:
        print(
            f"  capture[{window.name}] start={window.start_offset_ns:.1f} ns "
            f"length={window.length_ns:.1f} ns"
        )

In [ ]:
@dataclass(frozen=True)
class _CompatEvent:
    waveform_name: str
    start_offset_ns: float
    gain: float
    phase_offset_deg: float


@dataclass(frozen=True)
class _CompatCaptureWindow:
    name: str
    start_offset_ns: float
    length_ns: float


@dataclass(frozen=True)
class _CompatWaveform:
    sampling_period_ns: float
    iq_array: np.ndarray


class _CompatSequencer:
    def __init__(self, default_sampling_period_ns: float):
        self._waveform_library: dict[str, _CompatWaveform] = {}
        self._alias_to_events: dict[str, list[_CompatEvent]] = defaultdict(list)
        self._alias_to_capwin: dict[str, list[_CompatCaptureWindow]] = defaultdict(list)
        self._length_ns = 0.0
        self._default_sampling_period_ns = float(default_sampling_period_ns)

    def register_waveform(
        self, name: str, waveform, sampling_period_ns: float | None = None
    ):
        dt = (
            self._default_sampling_period_ns
            if sampling_period_ns is None
            else float(sampling_period_ns)
        )
        iq_array = np.asarray(waveform, dtype=np.complex128)
        if np.any(np.abs(iq_array) > 1.0):
            raise ValueError("The amplitude must be in the range -1 to 1.")
        self._waveform_library[name] = _CompatWaveform(dt, iq_array)

    def add_event(
        self,
        instrument_alias: str,
        waveform_name: str,
        start_offset_ns: float,
        gain: float = 1.0,
        phase_offset_deg: float = 0.0,
    ):
        if waveform_name not in self._waveform_library:
            raise ValueError(f"waveform '{waveform_name}' is not registered.")
        self._alias_to_events[instrument_alias].append(
            _CompatEvent(
                waveform_name=waveform_name,
                start_offset_ns=float(start_offset_ns),
                gain=float(gain),
                phase_offset_deg=float(phase_offset_deg),
            )
        )
        waveform = self._waveform_library[waveform_name]
        end_at_ns = (
            float(start_offset_ns)
            + len(waveform.iq_array) * waveform.sampling_period_ns
        )
        self._length_ns = max(self._length_ns, end_at_ns)

    def add_capture_window(
        self,
        instrument_alias: str,
        window_name: str,
        start_offset_ns: float,
        length_ns: float,
    ):
        self._alias_to_capwin[instrument_alias].append(
            _CompatCaptureWindow(
                name=window_name,
                start_offset_ns=float(start_offset_ns),
                length_ns=float(length_ns),
            )
        )
        self._length_ns = max(
            self._length_ns, float(start_offset_ns) + float(length_ns)
        )

    @property
    def length_ns(self) -> float:
        return self._length_ns

    def export_set_fixed_timeline_directive(
        self, instrument_alias: str, sampling_period_fs: int
    ):
        name_to_index: dict[str, int] = {}
        local_library = []
        local_events = []

        for event in self._alias_to_events[instrument_alias]:
            if event.waveform_name in name_to_index:
                index = name_to_index[event.waveform_name]
            else:
                waveform = self._waveform_library[event.waveform_name]
                index = len(local_library)
                name_to_index[event.waveform_name] = index
                local_library.append(
                    SimpleNamespace(
                        sampling_period_fs=round(waveform.sampling_period_ns * 1e6),
                        iq_array=waveform.iq_array,
                    )
                )

            local_events.append(
                SimpleNamespace(
                    waveform_index=index,
                    start_offset_samples=round(
                        event.start_offset_ns * 1e6 / sampling_period_fs
                    ),
                    gain=event.gain,
                    phase_offset_deg=event.phase_offset_deg,
                )
            )

        local_capture_windows = [
            SimpleNamespace(
                name=window.name,
                start_offset_samples=round(
                    window.start_offset_ns * 1e6 / sampling_period_fs
                ),
                length_samples=round(window.length_ns * 1e6 / sampling_period_fs),
            )
            for window in self._alias_to_capwin[instrument_alias]
        ]

        return SimpleNamespace(
            waveform_library=local_library,
            events=local_events,
            capture_windows=local_capture_windows,
            length=round(self.length_ns * 1e6 / sampling_period_fs),
        )


def _load_sequencer_class() -> tuple[type, str]:
    try:
        from quelware_client.client.helpers.sequencer import Sequencer
    except Exception as exc:
        return (
            _CompatSequencer,
            f"Using compatibility sequencer: {type(exc).__name__}: {exc}",
        )
    else:
        return Sequencer, "Loaded Sequencer from installed quelware_client package."


SequencerClass, sequencer_note = _load_sequencer_class()
print(sequencer_note)

In [ ]:
compiler = Quel3SequencerCompiler()
sequencer = compiler.compile(
    payload=payload,
    sequencer_factory=SequencerClass,
    default_sampling_period_ns=DT_NS,
)

fig_sequencer = plot_sequencer_timeline(
    sequencer,
    title="Sequencer Timeline (After Compiler)",
)
fig_sequencer.show()

sequencer_state = vars(sequencer)
print("registered waveforms:", list(sequencer_state["_waveform_library"].keys()))

In [ ]:
sequencer_state = vars(sequencer)
ctrl_00_name = sequencer_state["_alias_to_events"]["ctrl-00"][0].waveform_name
ctrl_01_name = sequencer_state["_alias_to_events"]["ctrl-01"][0].waveform_name
print("shared waveform for ctrl-00 and ctrl-01:", ctrl_00_name == ctrl_01_name)
print("ctrl-00 waveform:", ctrl_00_name)
print("ctrl-01 waveform:", ctrl_01_name)

directive_ctrl_00 = sequencer.export_set_fixed_timeline_directive(
    "ctrl-00",
    sampling_period_fs=400_000,
)
directive_readout = sequencer.export_set_fixed_timeline_directive(
    "readout-00",
    sampling_period_fs=READOUT_EXPORT_SAMPLING_PERIOD_FS,
)

print(
    "ctrl-00 directive: events=",
    len(directive_ctrl_00.events),
    "waveforms=",
    len(directive_ctrl_00.waveform_library),
    "length_samples=",
    directive_ctrl_00.length,
)
print(
    "readout directive: events=",
    len(directive_readout.events),
    "captures=",
    len(directive_readout.capture_windows),
    "length_samples=",
    directive_readout.length,
)

## What to check

- The control channels (`ctrl-00`, `ctrl-01`) reuse the same waveform shape.
- Scale/phase differences are encoded in `add_event(..., gain, phase_offset_deg)`.
- `PulseSchedule` currently uses a common dt, so payload event `sampling_period_ns` is `0.4` for all channels in this example.
- Readout conversion is still demonstrated by exporting the readout directive with `sampling_period_fs=800_000`.
- Multiple capture windows (`capture_0`, `capture_1`) overlap each readout round on the readout channel (`RQ00`).
- Capture windows are independent timeline entries and appear as separate lanes in the visualizer.
